# AI2-THOR：場景多視角與物件座標
依照 [官方 Full Starter Template](https://colab.research.google.com/github/allenai/ai2thor-colab/blob/main/templates/AI2_THOR_Full_Starter_Template.ipynb) 的寫法，使用 ai2thor-colab 啟動 X server、建立 Controller，並用 plot_frames / show_objects_table 顯示結果。

只需上傳這個檔案，選擇 Colab GPU 執行階段，依序執行。
若執行過前一版，先下載需要保留的結果，再「中斷連線並刪除執行階段」後重新連線，避免舊設定影響。

功能：安裝啟動、選擇場景、匯出全部物件、3／4 個不同位置朝中央拍攝、下載 ZIP。


## Installation
沿用官方範本的 Python 套件與 start_xserver() 流程。官方 helper 使用舊版 Xvfb 套件且不等待就緒，因此加入 X11／GLX 檢查。
若官方啟動失敗，自動安裝目前系統套件庫的 Xvfb，在空閒 display 啟動並保留日誌。備援使用 Mesa 軟體 OpenGL，可能比 GPU 渲染慢。
[官方 helper 原始碼](https://github.com/allenai/ai2thor-colab/blob/main/ai2thor_colab/__init__.py)


In [ ]:
!pip install --upgrade ai2thor ai2thor-colab


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
import ai2thor
import ai2thor_colab
from ai2thor.controller import Controller
from ai2thor_colab import plot_frames, show_objects_table

# Stop Unity before changing its display connection.
if globals().get('controller') is not None:
    try:
        controller.stop()
    except Exception as exc:
        print('Previous Controller cleanup:', exc)
    finally:
        controller = None

# Query the server extension list: has_extension() also requires client-side support.
# Run the probe in a child process so a broken server cannot hang the notebook.
def x11_ready(display):
    env = dict(os.environ, DISPLAY=display)
    code = "from Xlib.display import Display; d=Display(); assert 'GLX' in d.list_extensions(), 'X server does not advertise GLX'; d.sync(); d.close()"
    try:
        result = subprocess.run([sys.executable, '-c', code], env=env,
                                capture_output=True, text=True, timeout=4)
        return result.returncode == 0, result.stderr
    except subprocess.TimeoutExpired:
        return False, 'X11 probe timed out'

def wait_for_x11(display, process=None, seconds=20):
    deadline = time.monotonic() + seconds
    error = ''
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            return False, f'Xvfb exited with code {process.returncode}'
        ready, error = x11_ready(display)
        if ready:
            return True, ''
        time.sleep(0.5)
    return False, error

ready, detail = x11_ready(os.environ.get('DISPLAY', ':1'))
if not ready:
    # Official template startup. Its progress bar does not establish readiness.
    try:
        ai2thor_colab.start_xserver()
    except Exception as exc:
        print('Official startup:', exc)
    ready, detail = wait_for_x11(os.environ.get('DISPLAY', ':1'))

if not ready:
    print('Official X server unavailable; installing distribution Xvfb. Last error:', detail)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '--reinstall',
                    'xvfb', 'xauth', 'mesa-utils', 'libgl1-mesa-dri'], check=True)
    # Use a free display; leave unrelated X servers and their lock files alone.
    display_number = next((i for i in range(90, 120)
                           if not Path(f'/tmp/.X{i}-lock').exists()
                           and not Path(f'/tmp/.X11-unix/X{i}').exists()), None)
    if display_number is None:
        raise RuntimeError('No free display in :90..:119')
    display = f':{display_number}'
    x_log = Path('/content/ai2thor_xvfb.log')
    with x_log.open('w') as log:
        _thor_xvfb_process = subprocess.Popen(
            ['Xvfb', display, '-screen', '0', '1024x768x24', '-ac',
             '+extension', 'GLX', '+render', '-noreset', '-nolisten', 'tcp'],
            stdout=log, stderr=subprocess.STDOUT)
    ready, detail = wait_for_x11(display, _thor_xvfb_process)
    if not ready:
        if _thor_xvfb_process.poll() is None:
            _thor_xvfb_process.terminate()
            _thor_xvfb_process.wait(timeout=5)
        print(x_log.read_text(errors='replace'))
        raise RuntimeError(f'Xvfb startup failed: {detail}. Log: {x_log}')
    os.environ['DISPLAY'] = display
    # Xvfb supplies a virtual screen; Mesa supplies its software OpenGL renderer.
    os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'
    result = subprocess.run(['glxinfo', '-B'], capture_output=True, text=True, timeout=20)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        print(x_log.read_text(errors='replace'))
        raise RuntimeError('X11 is ready but OpenGL initialization failed; provide the output above.')

print('X11 / GLX ready:', os.environ.get('DISPLAY', ':1'))
print('AI2-THOR Version:', ai2thor.__version__)


In [ ]:
ready, detail = x11_ready(os.environ.get('DISPLAY', ':1'))
if not ready:
    raise RuntimeError(f'X server is no longer available; rerun the startup cell. {detail}')
if globals().get('controller') is not None:
    controller.stop()
    controller = None
try:
    controller = Controller()
except Exception:
    log = Path('/content/ai2thor_xvfb.log')
    if log.exists():
        print(log.read_text(errors='replace')[-12000:])
    raise


## Workspace：選擇場景
預設 FloorPlan1、4 張照片，可改為 3 張。HORIZON=0 為水平，正值向下、負值向上。每次重跑此格會重置場景並建立新的輸出資料夾。


In [ ]:
SCENE = 'FloorPlan1' #@param {type:"string"}
NUM_VIEWS = 4 #@param [3, 4] {type:"raw"}
HORIZON = 0 #@param {type:"slider", min:-30, max:60, step:15}

from pathlib import Path
from datetime import datetime, timezone
import copy
import json
import math
import csv
from PIL import Image

if NUM_VIEWS not in (3, 4):
    raise ValueError('NUM_VIEWS must be 3 or 4')
controller.reset(scene=SCENE, width=800, height=800,
                 gridSize=0.25, snapToGrid=False)
if not controller.last_event.metadata['lastActionSuccess']:
    raise RuntimeError(controller.last_event.metadata['errorMessage'])

output_dir = Path('/content/ai2thor_outputs') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_dir.mkdir(parents=True, exist_ok=False)
print('Output directory:', output_dir)
plot_frames(controller.last_event)


## Show Objects Table：所有物件的座標與朝向
保存全部 metadata objects，包含目前不可見的物件。position XYZ 為世界座標（公尺），Y 向上；rotation XYZ 為 API 回傳的 Euler 角度（度），不代表所有物件都有語意上的正面。
範圍是 AI2-THOR 標註物件，不含未標註的 Unity 幾何。完整 metadata 存 JSON，座標與角度另存 CSV。


In [ ]:
event = controller.step(action='ResetObjectFilter')
if not event.metadata['lastActionSuccess']:
    raise RuntimeError(event.metadata['errorMessage'])
objects = copy.deepcopy(event.metadata['objects'])
show_objects_table(objects)

(output_dir / 'objects.json').write_text(
    json.dumps(objects, ensure_ascii=False, indent=2), encoding='utf-8')
fields = ['objectId', 'objectType', 'name'] + [f'{field}_{axis}' for field in ('position', 'rotation') for axis in 'xyz']
with (output_dir / 'objects.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for obj in objects:
        row = {key: obj.get(key) for key in ('objectId', 'objectType', 'name')}
        for field in ('position', 'rotation'):
            row.update({f'{field}_{axis}': obj[field][axis] for axis in 'xyz'})
        writer.writerow(row)
print(f'Exported {len(objects)} objects.')


## 不同位置朝中央拍攝 3／4 張圖片
取得可到達位置，以其 XZ 外接矩形的角落為目標，選擇最近且不重複的位置，面向矩形中央。
真實角落可能被家具占據，因此使用靠近角落的可到達點；視線仍可能被家具遮擋。
傳送失敗會嘗試下一個候選點，圖片使用機器人的第一人稱相機。


In [ ]:
event = controller.step(action='GetReachablePositions')
if not event.metadata['lastActionSuccess']:
    raise RuntimeError(event.metadata['errorMessage'])
positions = event.metadata['actionReturn']
positions = list({tuple(p[a] for a in 'xyz'): p for p in positions}.values())
if len(positions) < NUM_VIEWS:
    raise RuntimeError('Not enough views; complete capture or choose another scene.')

xmin, xmax = min(p['x'] for p in positions), max(p['x'] for p in positions)
zmin, zmax = min(p['z'] for p in positions), max(p['z'] for p in positions)
center_x, center_z = (xmin + xmax) / 2, (zmin + zmax) / 2
corners = [(xmin, zmin), (xmax, zmin), (xmax, zmax), (xmin, zmax)]
used = set()
views = []
frames = []
for index, (corner_x, corner_z) in enumerate(corners[:NUM_VIEWS], 1):
    candidates = sorted(positions, key=lambda p: (p['x'] - corner_x)**2 + (p['z'] - corner_z)**2)
    last_error = ''
    for position in candidates:
        key = tuple(round(position[a], 4) for a in 'xyz')
        if key in used:
            continue
        yaw = math.degrees(math.atan2(center_x - position['x'], center_z - position['z'])) % 360
        event = controller.step(action='Teleport', position=position,
                                rotation=dict(x=0, y=yaw, z=0), horizon=HORIZON, standing=True)
        if not event.metadata['lastActionSuccess']:
            last_error = event.metadata['errorMessage']
            continue
        agent = copy.deepcopy(event.metadata['agent'])
        actual_key = tuple(round(agent['position'][a], 4) for a in 'xyz')
        if actual_key in used:
            continue
        used.add(actual_key)
        filename = f'view_{index:02d}.png'
        Image.fromarray(event.frame).save(output_dir / filename)
        frames.append(event.frame.copy())
        views.append(dict(image=filename, agent=agent,
                          cameraPosition=event.metadata.get('cameraPosition'),
                          fov=event.metadata.get('fov')))
        print(filename, 'position:', agent['position'], 'rotation:', agent['rotation'])
        plot_frames(event)
        break
    else:
        raise RuntimeError(f'Could not capture view {index}: {last_error}')

manifest = dict(scene=SCENE, num_views=len(views), ai2thor_version=ai2thor.__version__,
                position_unit='meter', rotation_unit='degree',
                object_snapshot='before camera teleports', object_count=len(objects),
                look_at_center_xz=dict(x=center_x, z=center_z), views=views)
(output_dir / 'views.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(output_dir / 'reachable_positions.json').write_text(json.dumps(positions, indent=2), encoding='utf-8')


## 拍攝位置與方向
灰點為可到達位置，箭頭為機器人朝向，紅色叉號為中央。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter([p['x'] for p in positions], [p['z'] for p in positions], s=8, color='lightgray')
ax.scatter(center_x, center_z, marker='x', color='red')
for i, view in enumerate(views, 1):
    p = view['agent']['position']
    yaw = math.radians(view['agent']['rotation']['y'])
    ax.quiver(p['x'], p['z'], math.sin(yaw), math.cos(yaw), angles='xy', scale_units='xy', scale=2)
    ax.annotate(str(i), (p['x'], p['z']))
ax.set(xlabel='X (m)', ylabel='Z (m)', title=SCENE)
ax.set_aspect('equal')
fig.savefig(output_dir / 'viewpoints.png', dpi=150, bbox_inches='tight')
plt.show()


## 下載結果
ZIP 包含照片、物件 CSV／JSON、每張圖的機器人 pose、可到達位置與方向圖。


In [ ]:
import shutil
from google.colab import files

if len(views) != NUM_VIEWS or not (output_dir / 'views.json').exists():
    raise RuntimeError('Not enough views; complete capture or choose another scene.')
archive = shutil.make_archive(str(output_dir), 'zip', root_dir=output_dir)
files.download(archive)


## 結束
完成或發生錯誤後可執行此格釋放模擬器。


In [ ]:
if globals().get('controller') is not None:
    controller.stop()
    controller = None


## 執行說明與參考
本版保留官方範本流程，補上 X server 就緒檢查及系統 Xvfb 備援。
如果仍啟動失敗，請提供「啟動環境」整格輸出，以及 /content/ai2thor_xvfb.log（若存在）。不要只提供 Controller 的最後一段 traceback。
已進行離線語法檢查，尚未在 Colab GPU／Unity 完整實測。

- [官方範本](https://colab.research.google.com/github/allenai/ai2thor-colab/blob/main/templates/AI2_THOR_Full_Starter_Template.ipynb)
- [物件 metadata](https://ai2thor.allenai.org/ithor/documentation/environment-state/)
- [導航](https://ai2thor.allenai.org/ithor/documentation/navigation/)
